In [ ]:
# Run this cell first — fixes table and equation alignment throughout the notebook
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 5 Lab — Data Cleaning and Visualisation
**Introduction to Python for Business Statistics**

---

### What This Lab Covers
- Loading data from a CSV file
- Identifying and handling missing data
- Detecting and correcting data quality issues
- Histograms — visualising distributions
- Boxplots — visualising spread and outliers
- Scatter plots — visualising relationships
- Correlation

**Estimated time:** 75–90 minutes

---

## Part 1 — Loading and Inspecting Data

In real work, data comes from files — not dictionaries you type by hand. The most common format is CSV (comma-separated values). Pandas reads CSV files with a single line.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual style for all plots
sns.set_theme(style="whitegrid")

# For this lab we build a realistic quarterly sales dataset directly
# In practice you would use: df = pd.read_csv("filename.csv")

np.random.seed(42)
n = 120

df = pd.DataFrame({
    "rep":        np.random.choice(["Chen", "Patel", "Okafor", "Rivera", "Thompson"], n),
    "region":     np.random.choice(["Northeast", "West", "South", "Midwest"], n),
    "quarter":    np.random.choice(["Q1", "Q2", "Q3", "Q4"], n),
    "revenue":    np.round(np.random.normal(42000, 8000, n), 2),
    "units_sold": np.random.randint(10, 80, n),
    "discount_pct": np.round(np.random.uniform(0, 0.30, n), 3)
})

# Introduce some missing values and a data quality issue
df.loc[np.random.choice(df.index, 8, replace=False), "revenue"] = np.nan
df.loc[np.random.choice(df.index, 5, replace=False), "units_sold"] = np.nan
df.loc[np.random.choice(df.index, 3, replace=False), "discount_pct"] = -0.05  # invalid negative discount

print(df.shape)
df.head()

In [ ]:
# Get an overview of the DataFrame
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe().round(2)

---
## Part 2 — Missing Data

### 📊 Stats Connection — Missing Data

Missing data is nearly universal in real datasets. Before analysing, you need to know:
- How much data is missing?
- Is it missing at random, or is there a pattern?
- What should you do about it?

| Strategy | When to Use |
|:---------|:------------|
| **Drop rows** | Few missing values, missing at random |
| **Fill with mean/median** | Numerical data, small proportion missing |
| **Fill with mode** | Categorical data |
| **Fill with a constant** | When a specific value (e.g. 0) makes business sense |
| **Flag and investigate** | When the missing value itself may be meaningful |

In [ ]:
# Count missing values per column
print(df.isnull().sum())
print()

# As a percentage of total rows
print((df.isnull().sum() / len(df) * 100).round(1))

In [ ]:
# Fill missing revenue with the median (more robust than mean if outliers exist)
median_revenue = df["revenue"].median()
df["revenue"] = df["revenue"].fillna(median_revenue)

# Fill missing units_sold with the median
median_units = df["units_sold"].median()
df["units_sold"] = df["units_sold"].fillna(median_units)

print("Missing values after filling:")
print(df.isnull().sum())

---
### ✏️ Exercise 2.1 — Missing Data Investigation

Using the cleaned `df`:

In [ ]:
# Exercise 2.1

# 1. How many rows had missing revenue values before filling?
#    (Check the output from the isnull().sum() cell above)
#

# 2. Why did we use median rather than mean to fill missing revenue?
#    Write your answer as a comment.
#

# 3. What would happen to the mean revenue if we had filled missing values
#    with 0 instead of the median? Would it be higher or lower? Why?
#

---
## Part 3 — Data Quality

Missing values are one problem. Invalid values are another — data that is present but wrong. Common examples in business data:
- Negative prices or quantities
- Discount percentages above 100% or below 0%
- Future dates in historical records
- Duplicate rows

In [ ]:
# Find rows with invalid discount values (should be between 0 and 1)
invalid_discounts = df[df["discount_pct"] < 0]
print(f"Rows with invalid discount: {len(invalid_discounts)}")
print(invalid_discounts[["rep", "revenue", "discount_pct"]])

In [ ]:
# Replace invalid discounts with 0
df.loc[df["discount_pct"] < 0, "discount_pct"] = 0

# Verify the fix
print("Min discount after fix:", df["discount_pct"].min())
print("Max discount after fix:", df["discount_pct"].max())

In [ ]:
# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

---
### ✏️ Exercise 3.1 — Data Quality Checks

Add two more validation checks to the dataset.

In [ ]:
# Exercise 3.1

# 1. Find any rows where revenue is negative (invalid)
negative_revenue = 
print(f"Rows with negative revenue: {len(negative_revenue)}")

# 2. Find any rows where units_sold is zero or negative
invalid_units = 
print(f"Rows with invalid units_sold: {len(invalid_units)}")

# 3. Add a column called revenue_per_unit = revenue / units_sold
#    Handle the case where units_sold might be zero using np.where
#    Hint: np.where(condition, value_if_true, value_if_false)
df["revenue_per_unit"] = 
print(df[["revenue", "units_sold", "revenue_per_unit"]].head())

---
## Part 4 — Histograms

### 📊 Stats Connection — Distributions

A **histogram** divides a continuous variable into equal-width bins and shows how many values fall in each bin. It reveals the **shape** of a distribution:

| Shape | Description | What It Suggests |
|:------|:------------|:-----------------|
| **Symmetric / bell-shaped** | Values cluster around the centre | Mean ≈ median; use mean |
| **Right-skewed** | Long tail to the right | Mean > median; outliers on high end |
| **Left-skewed** | Long tail to the left | Mean < median; outliers on low end |
| **Bimodal** | Two peaks | Possibly two sub-groups in the data |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Revenue histogram
axes[0].hist(df["revenue"], bins=20, color="steelblue", edgecolor="white")
axes[0].axvline(df["revenue"].mean(),   color="red",    linestyle="--", label="Mean")
axes[0].axvline(df["revenue"].median(), color="orange", linestyle="--", label="Median")
axes[0].set_title("Revenue Distribution")
axes[0].set_xlabel("Revenue ($)")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Units sold histogram
axes[1].hist(df["units_sold"], bins=15, color="coral", edgecolor="white")
axes[1].set_title("Units Sold Distribution")
axes[1].set_xlabel("Units Sold")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

### Seaborn Histogram with KDE

Seaborn adds a **KDE** (kernel density estimate) — a smoothed curve showing the distribution shape.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["revenue"], bins=20, kde=True, color="steelblue", ax=ax)
ax.set_title("Revenue Distribution with KDE")
ax.set_xlabel("Revenue ($)")
plt.tight_layout()
plt.show()

---
### ✏️ Exercise 4.1 — Histogram Analysis

In [ ]:
# Exercise 4.1

# 1. Create a histogram of discount_pct with 15 bins
#    Add a vertical line for the mean discount
#    Label axes and add a title

fig, ax = plt.subplots(figsize=(8, 4))

# your code here

plt.tight_layout()
plt.show()

# 2. Describe the shape of the distribution in a comment.
#    Is the mean close to the median? What does that tell you?
print(f"Mean discount:   {df['discount_pct'].mean():.3f}")
print(f"Median discount: {df['discount_pct'].median():.3f}")
#

---
## Part 5 — Boxplots

### 📊 Stats Connection — The Five-Number Summary

A **boxplot** visualises the five-number summary of a dataset:

| Element | Value |
|:--------|:------|
| Minimum (excluding outliers) | Lower whisker |
| Q1 (25th percentile) | Bottom of box |
| Median (Q2) | Line inside box |
| Q3 (75th percentile) | Top of box |
| Maximum (excluding outliers) | Upper whisker |
| Outliers | Individual dots beyond the whiskers |

The box spans the IQR. Whiskers extend to the Tukey fences ($Q1 - 1.5 \times IQR$ and $Q3 + 1.5 \times IQR$). Points beyond the whiskers are plotted individually as outliers.

In [ ]:
# Boxplot of revenue by region
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="region", y="revenue", palette="Set2", hue="region", legend=False, ax=ax)
ax.set_title("Revenue by Region")
ax.set_xlabel("Region")
ax.set_ylabel("Revenue ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot of revenue by quarter — track seasonal patterns
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="quarter", y="revenue",
            order=["Q1", "Q2", "Q3", "Q4"], palette="Blues", ax=ax)
ax.set_title("Revenue by Quarter")
ax.set_xlabel("Quarter")
ax.set_ylabel("Revenue ($)")
plt.tight_layout()
plt.show()

---
### ✏️ Exercise 5.1 — Boxplot Analysis

In [ ]:
# Exercise 5.1

# 1. Create a boxplot comparing units_sold across regions
fig, ax = plt.subplots(figsize=(9, 5))

# your code here

plt.tight_layout()
plt.show()

# 2. Create a side-by-side figure: revenue boxplot by rep (left) and
#    units_sold boxplot by rep (right)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# your code here

plt.tight_layout()
plt.show()

# 3. Based on the boxplots, which rep shows the most variability in revenue?
#    Write your answer as a comment.
#

---
## Part 6 — Scatter Plots and Correlation

### 📊 Stats Connection — Correlation

A **scatter plot** shows the relationship between two continuous variables. Each point represents one observation.

**Correlation** measures the strength and direction of a linear relationship between two variables.

The **Pearson correlation coefficient** $r$ ranges from $-1$ to $+1$:

| Value | Interpretation |
|:------|:---------------|
| $r = +1$ | Perfect positive linear relationship |
| $r = 0$ | No linear relationship |
| $r = -1$ | Perfect negative linear relationship |
| $|r| > 0.7$ | Strong |
| $0.4 < |r| \leq 0.7$ | Moderate |
| $|r| \leq 0.4$ | Weak |

> **Correlation is not causation.** Two variables can be correlated without one causing the other.

In [ ]:
# Scatter plot: units_sold vs revenue
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x="units_sold", y="revenue",
                hue="region", alpha=0.7, ax=ax)
ax.set_title("Units Sold vs Revenue")
ax.set_xlabel("Units Sold")
ax.set_ylabel("Revenue ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Calculate correlation between units_sold and revenue
corr = df["units_sold"].corr(df["revenue"])
print(f"Correlation (units_sold vs revenue): {corr:.3f}")

In [ ]:
# Correlation matrix for all numeric columns
numeric_cols = df[["revenue", "units_sold", "discount_pct", "revenue_per_unit"]]
corr_matrix  = numeric_cols.corr().round(3)
print(corr_matrix)

In [ ]:
# Heatmap of correlation matrix
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0,
            fmt=".2f", linewidths=0.5, ax=ax)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

---
### ✏️ Exercise 6.1 — Scatter Plot and Correlation

In [ ]:
# Exercise 6.1

# 1. Create a scatter plot of discount_pct (x) vs revenue (y)
#    Colour points by quarter
fig, ax = plt.subplots(figsize=(8, 5))

# your code here

plt.tight_layout()
plt.show()

# 2. Calculate the correlation between discount_pct and revenue
corr_disc_rev = 
print(f"Correlation (discount_pct vs revenue): {corr_disc_rev:.3f}")

# 3. Interpret the result:
#    Is the relationship positive or negative? Strong, moderate, or weak?
#    Does this make business sense? Why or why not?
#

---
## Part 7 — Challenge Exercise

This section is optional.

Produce a complete visual summary of the dataset with four plots in a 2×2 grid:
1. Revenue histogram with mean and median lines
2. Boxplot of revenue by rep
3. Scatter plot of units_sold vs revenue coloured by region
4. Correlation heatmap of numeric variables

Then answer the following in comments:
- Which rep has the highest median revenue?
- Which two variables are most strongly correlated?
- Is the revenue distribution symmetric? How can you tell from both the histogram and the mean/median comparison?

In [ ]:
# Challenge — your code here
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1 — Revenue histogram (axes[0, 0])

# Plot 2 — Revenue boxplot by rep (axes[0, 1])

# Plot 3 — Scatter plot units_sold vs revenue (axes[1, 0])

# Plot 4 — Correlation heatmap (axes[1, 1])

plt.suptitle("Quarterly Sales Dashboard", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Answers:
# Highest median revenue rep:
# Most correlated variables:
# Revenue distribution shape:
#

---
## Week 5 Summary

| Concept | Key Point |
|:--------|:----------|
| `pd.read_csv()` | Load tabular data from a file |
| `df.isnull().sum()` | Count missing values per column |
| `fillna()` | Fill missing values — use median for skewed numeric data |
| Boolean filtering | `df[df["col"] < 0]` — find invalid values |
| `df.loc[condition, col] = value` | Fix invalid values in place |
| Histogram | Shows distribution shape — bins continuous data |
| KDE | Smoothed curve overlaid on histogram |
| Boxplot | Shows five-number summary and outliers — good for comparing groups |
| Scatter plot | Shows relationship between two continuous variables |
| Correlation $r$ | $-1$ to $+1$ — strength and direction of linear relationship |
| Heatmap | Visualises a full correlation matrix |

---
**Next week:** Descriptive statistics with Pandas (`.describe()`, `groupby`, pivot tables) and the micro-project.